# Behavior analysis — stay probability differences

Loads the per-subject stay probability produced by
`1_data_preprocessing.ipynb`, computes **diff = stay(common-rewarded) − stay(rare-rewarded)**
per subject per stage, plots it, and runs a one-way repeated-measures ANOVA
across stages 4.6 / 4.7 / 4.8 (within-subjects factor: training stage).

In [ ]:
# Setup: config, imports, session loader
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_here = os.path.abspath('')
_root = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here
sys.path.insert(0, _root)
sys.path.insert(0, os.path.join(_root, 'src'))

from config import (
    SUBJECTS, STAGES, raw_subject_dir, FIGURES_DIR, PROJECT_ROOT
)
from src.data_import import Experiment

STAY_STAGES = ['4.6', '4.7', '4.8']
BEHAVIOR_DIR = os.path.join(PROJECT_ROOT, 'results', 'behavior')
os.makedirs(BEHAVIOR_DIR, exist_ok=True)

print('Subjects:', SUBJECTS)
print('Stages  :', STAY_STAGES)
print('Output  :', BEHAVIOR_DIR)

In [ ]:
# Load sessions per (subject, stage)
# Stage 4.8: all sessions in the main folder (no filter).
# Stages 4.6/4.7: only _Training folder, filtered by training_stage.

def load_sessions(subject, stage):
    if stage == '4.8':
        folder = raw_subject_dir(subject, training=False)
        if not os.path.isdir(folder):
            return []
        sessions = list(Experiment(folder).sessions)
    else:
        folder = raw_subject_dir(subject, training=True)
        if not os.path.isdir(folder):
            return []
        sessions = [s for s in Experiment(folder).sessions
                    if s.training_stage == stage]
    seen, unique = set(), []
    for s in sorted(sessions, key=lambda s: s.datetime_string):
        if s.file_name not in seen:
            seen.add(s.file_name)
            unique.append(s)
    return unique

In [ ]:
# Parse one session into the arrays needed for stay probability
# Conventions (match 6_behavioral_simulation.ipynb Cell 4):
#   c  : 1=left, 2=right          ss : 1=up, 2=down
#   tt : 1=free choice, 2=forced  r  : 1.0=rewarded, 0.0=not
#   trans_type: 'A' or 'B' (from d[8][3] of the first trial)

def parse_session(session):
    b = session.print_lines
    T = len(b)
    trans_type = b[0].split(' ')[8][3]  # 'A' or 'B'
    c  = np.ones(T, dtype=int)
    ss = np.ones(T, dtype=int)
    tt = np.ones(T, dtype=int)
    r  = np.zeros(T)
    for j in range(T):
        d = b[j].split(' ')
        if d[4] == 'C:0': c[j]  = 2
        if d[5] == 'S:0': ss[j] = 2
        r[j] = 1.0 if d[6] == 'O:1' else 0.0
        if d[9] != 'CT:FC': tt[j] = 2
    return c, ss, tt, r, trans_type


def stay_prob_4cond(c, ss, tt, r, trans_type):
    """Return [rew_com, rew_rare, unrew_com, unrew_rare], NaN if no data.

    Only adjacent FC→FC trial pairs are counted.
    Type A: matching c==ss → common.  Type B: c!=ss → common.
    """
    stay  = np.zeros(4)
    total = np.zeros(4)
    for t in range(len(c) - 1):
        if tt[t] != 1 or tt[t + 1] != 1:
            continue
        if trans_type == 'A':
            is_common = (c[t] == ss[t])
        else:
            is_common = (c[t] != ss[t])
        rew_idx  = 0 if r[t] == 1.0 else 1
        comm_idx = 0 if is_common else 1
        cond     = rew_idx * 2 + comm_idx
        stay[cond]  += int(c[t + 1] == c[t])
        total[cond] += 1
    return np.where(total > 0, stay / total, np.nan)

In [ ]:
# Compute per-session stay probability -> save long-format CSV
COND_LABELS = ['rew_com', 'rew_rare', 'unrew_com', 'unrew_rare']

rows = []
for stage in STAY_STAGES:
    for subject in SUBJECTS:
        for sess_idx, session in enumerate(load_sessions(subject, stage)):
            c, ss, tt, r, ttype = parse_session(session)
            sp = stay_prob_4cond(c, ss, tt, r, ttype)
            rows.append({
                'subject':     subject,
                'stage':       stage,
                'session_idx': sess_idx,
                'date':        session.datetime_string,
                'trans_type':  ttype,
                'n_trials':    len(c),
                **{f'stay_{lab}': sp[i] for i, lab in enumerate(COND_LABELS)},
            })

df_sess = pd.DataFrame(rows)
sess_csv = os.path.join(BEHAVIOR_DIR, 'stay_prob_per_session.csv')
df_sess.to_csv(sess_csv, index=False)
print(f'Saved {len(df_sess)} session rows to {sess_csv}')
df_sess.head()

In [ ]:
# Collapse to per-subject (average across sessions) -> save CSV
stay_cols = [f'stay_{lab}' for lab in COND_LABELS]

df_subj = (df_sess.groupby(['stage', 'subject'])[stay_cols]
                  .mean()
                  .reset_index())
subj_csv = os.path.join(BEHAVIOR_DIR, 'stay_prob_per_subject.csv')
df_subj.to_csv(subj_csv, index=False)
print(f'Saved {len(df_subj)} subject-stage rows to {subj_csv}')
df_subj

In [ ]:
# Plot: one panel per stage, 4 bars (C1, R1, C0, R0)
#   Bars   = mean across subjects, error bars = SEM
#   Dots   = individual subjects (jittered)
#   Colors : green=common, orange=rare; lighter alpha=unrewarded

COND_DISPLAY = ['C1', 'R1', 'C0', 'R0']
COND_COLORS  = ['#10A551', '#F6921E', '#10A551', '#F6921E']
COND_ALPHA   = [1.0, 1.0, 0.5, 0.5]

fig, axes = plt.subplots(1, len(STAY_STAGES),
                         figsize=(3.2 * len(STAY_STAGES), 3.5),
                         sharey=True, dpi=150)
rng = np.random.default_rng(0)
x = np.arange(4)

for ax, stage in zip(axes, STAY_STAGES):
    subj = df_subj[df_subj['stage'] == stage].set_index('subject')[stay_cols]
    m = subj.mean(axis=0).values
    sem = subj.sem(axis=0).values

    for xi in x:
        ax.bar(xi, m[xi], yerr=sem[xi],
               color=COND_COLORS[xi], alpha=COND_ALPHA[xi],
               capsize=4, width=0.7,
               error_kw={'linewidth': 1.5})

    for _, row_vals in subj.iterrows():
        jitter = rng.uniform(-0.08, 0.08, size=4)
        ax.plot(x + jitter, row_vals.values,
                'o', color='k', ms=4, alpha=0.6, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels(COND_DISPLAY)
    ax.set_title(f'Stage {stage}  (n={subj.shape[0]})')
    ax.set_ylim(0, 1)
    ax.axhline(0.5, color='gray', lw=0.6, ls='--', alpha=0.5)

axes[0].set_ylabel('Stay probability')
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'stay_prob_experimental_stages.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

_here = os.path.abspath('')
_root = os.path.dirname(_here) if os.path.basename(_here) == 'notebooks' else _here
sys.path.insert(0, _root)

from config import FIGURES_DIR, PROJECT_ROOT

BEHAVIOR_DIR = os.path.join(PROJECT_ROOT, 'results', 'behavior')
SUBJ_CSV = os.path.join(BEHAVIOR_DIR, 'stay_prob_per_subject.csv')

if not os.path.exists(SUBJ_CSV):
    raise FileNotFoundError(
        f'{SUBJ_CSV} not found — run 1_data_preprocessing.ipynb first.')

df = pd.read_csv(SUBJ_CSV)
df['stage'] = df['stage'].astype(str)
STAGES = ['4.6', '4.7', '4.8']
df = df[df['stage'].isin(STAGES)].copy()
print(df)

In [ ]:
# Compute diff = stay(common-rewarded) − stay(rare-rewarded) per subject per stage
df['diff_C1_R1'] = df['stay_rew_com'] - df['stay_rew_rare']

# Pivot to a subject x stage matrix (required for repeated-measures ANOVA)
pivot = df.pivot(index='subject', columns='stage', values='diff_C1_R1')[STAGES]
# Drop subjects missing any stage (rmANOVA requires balanced design)
pivot = pivot.dropna()
print(f'Subjects with data at all {len(STAGES)} stages: {pivot.shape[0]}')
pivot

In [ ]:
# One-way repeated-measures ANOVA (within-subjects factor: stage)
#   Y_ij = subject i, stage j
#   SS_total       = sum over all cells of (Y - grand_mean)^2
#   SS_stage       = n_subj * sum_j (mean_j - grand_mean)^2
#   SS_subj        = n_stage * sum_i (mean_i - grand_mean)^2
#   SS_error       = SS_total - SS_stage - SS_subj
#   df_stage       = k - 1,  df_subj = n - 1,  df_error = (k-1)(n-1)
#   F = MS_stage / MS_error; p from F(df_stage, df_error)

def rm_anova_oneway(Y):
    """Y : ndarray (n_subjects, k_conditions), balanced, no NaNs."""
    n, k = Y.shape
    grand = Y.mean()
    cond_means = Y.mean(axis=0)
    subj_means = Y.mean(axis=1)

    ss_total = ((Y - grand) ** 2).sum()
    ss_cond  = n * ((cond_means - grand) ** 2).sum()
    ss_subj  = k * ((subj_means - grand) ** 2).sum()
    ss_error = ss_total - ss_cond - ss_subj

    df_cond, df_subj, df_error = k - 1, n - 1, (k - 1) * (n - 1)
    ms_cond  = ss_cond  / df_cond
    ms_error = ss_error / df_error
    F = ms_cond / ms_error
    p = 1 - stats.f.cdf(F, df_cond, df_error)

    # partial eta-squared (effect size for within-subjects factor)
    eta2_p = ss_cond / (ss_cond + ss_error)

    return {
        'n_subjects': n, 'k_conditions': k,
        'SS_stage': ss_cond, 'SS_subject': ss_subj,
        'SS_error': ss_error, 'SS_total': ss_total,
        'df_stage': df_cond, 'df_error': df_error,
        'MS_stage': ms_cond, 'MS_error': ms_error,
        'F': F, 'p': p, 'partial_eta2': eta2_p,
    }

res = rm_anova_oneway(pivot.values)
print('Repeated-measures ANOVA  (factor: stage)')
print('-' * 50)
for key, val in res.items():
    if isinstance(val, (int, np.integer)):
        print(f'  {key:<14} = {val}')
    else:
        print(f'  {key:<14} = {val:.5f}')

print()
print(f'F({res["df_stage"]}, {res["df_error"]}) = {res["F"]:.3f},'
      f'  p = {res["p"]:.4f},'
      f'  partial η² = {res["partial_eta2"]:.3f}')

In [ ]:
# Mauchly-style sphericity check + Greenhouse-Geisser correction
# (useful when k>=3 and sphericity may be violated)

def greenhouse_geisser(Y):
    """Return GG epsilon and the GG-corrected p-value."""
    n, k = Y.shape
    # Covariance of the k conditions across subjects
    S = np.cov(Y, rowvar=False, ddof=1)
    # Orthonormal contrast matrix for k conditions (centered Helmert-like)
    # A simpler route: epsilon = (trace S_centered)^2 / ((k-1) * sum(S_centered^2))
    # where S_centered is S with row/column means subtracted.
    row_mean = S.mean(axis=1, keepdims=True)
    col_mean = S.mean(axis=0, keepdims=True)
    grand    = S.mean()
    Sc = S - row_mean - col_mean + grand
    num = np.trace(Sc) ** 2
    den = (k - 1) * np.sum(Sc ** 2)
    eps = num / den if den > 0 else 1.0
    eps = min(1.0, max(1.0 / (k - 1), eps))
    return eps

Y = pivot.values
eps = greenhouse_geisser(Y)
df_s_gg = res['df_stage'] * eps
df_e_gg = res['df_error'] * eps
p_gg = 1 - stats.f.cdf(res['F'], df_s_gg, df_e_gg)
print(f'Greenhouse-Geisser epsilon = {eps:.3f}')
print(f'GG-corrected F({df_s_gg:.2f}, {df_e_gg:.2f}) = {res["F"]:.3f},  p = {p_gg:.4f}')

In [ ]:
# Pairwise paired t-tests between stages (Holm correction)
from itertools import combinations


def _stars(p):
    """Significance marker from a p-value (ns if not significant)."""
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


pairs = list(combinations(STAGES, 2))
pvals_raw = []
rows = []
for a, b in pairs:
    t, p = stats.ttest_rel(pivot[a], pivot[b])
    pvals_raw.append(p)
    rows.append({'pair': f'{a} vs {b}',
                 't': t, 'p_raw': p,
                 'mean_diff': pivot[a].mean() - pivot[b].mean()})

# Holm-Bonferroni
order = np.argsort(pvals_raw)
m = len(pvals_raw)
p_holm = np.empty(m)
running_max = 0.0
for rank, idx in enumerate(order):
    adj = pvals_raw[idx] * (m - rank)
    running_max = max(running_max, min(adj, 1.0))
    p_holm[idx] = running_max

for r, p_adj in zip(rows, p_holm):
    r['p_holm'] = p_adj
    r['sig'] = _stars(p_adj)          # star markers from Holm-corrected p

post_hoc = pd.DataFrame(rows)
print(post_hoc.to_string(index=False))

In [ ]:
# Plot: per-subject diff across stages + mean ± SEM
# Descriptive stage labels for the paper figure (match manuscript wording)
STAGE_LABELS = {'4.6': 'Early', '4.7': 'Intermediate', '4.8': 'Late'}
TICK_FONTSIZE = 14   # x-/y-axis tick label size

fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
x = np.arange(len(STAGES))

# Per-subject lines
for subj in pivot.index:
    ax.plot(x, pivot.loc[subj].values, '-o',
            color='gray', alpha=0.5, ms=4, lw=0.8)

# Mean ± SEM
m = pivot.mean(axis=0).values
sem = pivot.sem(axis=0).values
ax.errorbar(x, m, yerr=sem, fmt='s-', color='black',
            capsize=5, capthick=1.5, lw=2, ms=7,
            label='mean ± SEM')

ax.set_xticks(x)
ax.set_xticklabels([STAGE_LABELS.get(s, s) for s in STAGES])
# ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)  # larger tick labels
# ax.set_xlabel('Training stage')
# ax.set_ylabel('Stay prob diff  (rew-common − rew-rare)')
ax.axhline(0, color='gray', lw=0.6, ls='--', alpha=0.6)

title = (f'rmANOVA: F({res["df_stage"]}, {res["df_error"]})'
         f' = {res["F"]:.2f},  p = {res["p"]:.4f}')
# ax.set_title(title, fontsize=10)
# ax.legend(fontsize=9, loc='best')
plt.tight_layout()

# erase right and top spines for cleaner look
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)

fig_path = os.path.join(FIGURES_DIR, 'stay_diff_C1minusR1_rmANOVA.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

## Better-choice rate — optimal first-step choices across stages

Fraction of **free-choice** trials on which the mouse selected the first-step
action whose *common* transition leads to the currently-rewarded second-step
state. Neutral blocks (50/50) are excluded. Uses the same one-way
repeated-measures ANOVA across stages 4.6 / 4.7 / 4.8 as the stay-probability
analysis above, followed by **Bonferroni-corrected** pairwise paired t-tests
(the Fig 2A convention).

In [ ]:
# Better-choice rate per session -> per subject.
# "Better" choice = the first-step action whose COMMON transition most often
# reaches the currently-rewarded second-step state.  Neutral blocks are excluded.
# Block state from d[8][2]: D=down rewarded (0), U=up rewarded (1), N=neutral (2).
BLOCK_CHAR = {'D': 0, 'U': 1, 'N': 2}


def parse_better_choice(session):
    """Return c (1=left,2=right), tt (1=free,2=forced), B (block), PR, PL."""
    b = session.print_lines
    T = len(b)
    trans_A = (b[0].split(' ')[8][3] == 'A')
    if trans_A:
        PL, PR = np.array([0.8, 0.2]), np.array([0.2, 0.8])
    else:
        PL, PR = np.array([0.2, 0.8]), np.array([0.8, 0.2])
    c  = np.ones(T, dtype=int)
    tt = np.ones(T, dtype=int)
    B  = np.zeros(T, dtype=int)
    for j in range(T):
        d = b[j].split(' ')
        if d[4] == 'C:0':   c[j]  = 2
        if d[9] != 'CT:FC': tt[j] = 2
        B[j] = BLOCK_CHAR.get(d[8][2], 2)
    return c, tt, B, PR, PL


def _correct_choice(block_state, PR, PL):
    """Optimal first-step choice (1=left, 2=right); None for neutral block."""
    if block_state == 0:                 # down (ss=2) rewarded
        return 1 if PL[1] > PR[1] else 2
    if block_state == 1:                 # up (ss=1) rewarded
        return 1 if PL[0] > PR[0] else 2
    return None                          # neutral (50/50)


def better_choice_rate(session):
    """Fraction of free-choice, non-neutral trials taking the optimal choice."""
    c, tt, B, PR, PL = parse_better_choice(session)
    n = hit = 0
    for j in range(len(c)):
        if tt[j] != 1:                   # free-choice trials only
            continue
        cc = _correct_choice(B[j], PR, PL)
        if cc is None:                   # skip neutral blocks
            continue
        n   += 1
        hit += int(c[j] == cc)
    return (hit / n if n > 0 else np.nan), n


bc_rows = []
for stage in STAGES:
    for subject in SUBJECTS:
        for sess_idx, session in enumerate(load_sessions(subject, stage)):
            rate, n_fc = better_choice_rate(session)
            bc_rows.append({'stage': stage, 'subject': subject,
                            'session_idx': sess_idx,
                            'date': session.datetime_string,
                            'better_rate': rate, 'n_fc': n_fc})

df_bc_sess = pd.DataFrame(bc_rows)
bc_sess_csv = os.path.join(BEHAVIOR_DIR, 'better_choice_per_session.csv')
df_bc_sess.to_csv(bc_sess_csv, index=False)

df_bc_subj = (df_bc_sess.groupby(['stage', 'subject'])['better_rate']
                        .mean().reset_index())
bc_subj_csv = os.path.join(BEHAVIOR_DIR, 'better_choice_per_subject.csv')
df_bc_subj.to_csv(bc_subj_csv, index=False)
print(f'Saved {len(df_bc_sess)} session rows -> {bc_sess_csv}')
print(f'Saved {len(df_bc_subj)} subject rows  -> {bc_subj_csv}')

# Subject x stage matrix (balanced design required for rmANOVA)
bc_pivot = (df_bc_subj.pivot(index='subject', columns='stage', values='better_rate')
                      [STAGES].dropna())
print(f'\nSubjects with data at all {len(STAGES)} stages: {bc_pivot.shape[0]}')
print('Stage means:', bc_pivot.mean().round(3).to_dict())
bc_pivot

In [ ]:
# One-way repeated-measures ANOVA on better-choice rate (reuses rm_anova_oneway)
# + Bonferroni-corrected pairwise paired t-tests (matches the Fig 2A convention).
from itertools import combinations


def _stars(p):
    """Significance marker from a p-value (ns if not significant)."""
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'


res_bc = rm_anova_oneway(bc_pivot.values)
print('Better-choice rate — repeated-measures ANOVA (factor: stage)')
print('-' * 55)
print(f'F({res_bc["df_stage"]}, {res_bc["df_error"]}) = {res_bc["F"]:.3f},'
      f'  p = {res_bc["p"]:.5f}  {_stars(res_bc["p"])},'
      f'  partial eta^2 = {res_bc["partial_eta2"]:.3f}')

pairs = list(combinations(STAGES, 2))
m = len(pairs)
print('\nPairwise paired t-tests (Bonferroni-corrected):')
bc_posthoc = []
for a, b in pairs:
    t, p = stats.ttest_rel(bc_pivot[a], bc_pivot[b])
    p_bonf = min(p * m, 1.0)
    sig = _stars(p_bonf)
    bc_posthoc.append({'pair': f'{a} vs {b}', 't': t,
                       'p_raw': p, 'p_bonf': p_bonf, 'sig': sig})
    print(f'  {a} vs {b}: t({bc_pivot.shape[0] - 1}) = {t:+.2f},'
          f'  p = {p:.4f},  p_bonf = {p_bonf:.4f}  {sig}')
bc_posthoc = pd.DataFrame(bc_posthoc)

In [ ]:
# Plot: better-choice rate across stages (per-subject lines + mean ± SEM)
from matplotlib.ticker import PercentFormatter

TICK_FONTSIZE = 14   # y-/x-axis tick label size

# Descriptive stage labels for the paper figure (match manuscript wording)
STAGE_LABELS = {'4.6': 'Early', '4.7': 'Intermediate', '4.8': 'Late'}

fig, ax = plt.subplots(figsize=(4,4), dpi=300)
x = np.arange(len(STAGES))

# Per-subject lines (values as percentages)
for subj in bc_pivot.index:
    ax.plot(x, bc_pivot.loc[subj].values * 100, '-o',
            color='gray', alpha=0.5, ms=4, lw=0.8)

# Mean ± SEM (as percentages)
m_bc   = bc_pivot.mean(axis=0).values * 100
sem_bc = bc_pivot.sem(axis=0).values * 100
ax.errorbar(x, m_bc, yerr=sem_bc, fmt='s-', color='black',
            capsize=5, capthick=1.5, lw=2, ms=7, label='mean ± SEM')

ax.axhline(50, color='gray', lw=0.6, ls='--', alpha=0.6)  # chance (50%)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.set_xticks(x)
ax.set_xticklabels([STAGE_LABELS.get(s, s) for s in STAGES])
ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)  # larger tick labels
# ax.set_xlabel('Training stage')
# ax.set_ylabel('Better-choice rate')

title = (f'rmANOVA: F({res_bc["df_stage"]}, {res_bc["df_error"]})'
         f' = {res_bc["F"]:.2f},  p = {res_bc["p"]:.4f}')
# ax.set_title(title, fontsize=10)
# ax.legend(fontsize=9, loc='best')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'better_choice_rate_rmANOVA.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')

In [ ]:
# Plot: better-choice rate across stages (per-subject lines + mean ± SEM)
from matplotlib.ticker import PercentFormatter

TICK_FONTSIZE = 14   # y-/x-axis tick label size

fig, ax = plt.subplots(figsize=(4,4), dpi=300)
x = np.arange(len(STAGES))

# Per-subject lines (values as percentages)
for subj in bc_pivot.index:
    ax.plot(x, bc_pivot.loc[subj].values * 100, '-o',
            color='gray', alpha=0.5, ms=4, lw=0.8)

# Mean ± SEM (as percentages)
m_bc   = bc_pivot.mean(axis=0).values * 100
sem_bc = bc_pivot.sem(axis=0).values * 100
ax.errorbar(x, m_bc, yerr=sem_bc, fmt='s-', color='black',
            capsize=5, capthick=1.5, lw=2, ms=7, label='mean ± SEM')

ax.axhline(50, color='gray', lw=0.6, ls='--', alpha=0.6)  # chance (50%)
ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
ax.set_xticks(x)
ax.set_xticklabels(STAGES)
ax.tick_params(axis='both', labelsize=TICK_FONTSIZE)  # larger tick labels
# ax.set_xlabel('Training stage')
# ax.set_ylabel('Better-choice rate')

title = (f'rmANOVA: F({res_bc["df_stage"]}, {res_bc["df_error"]})'
         f' = {res_bc["F"]:.2f},  p = {res_bc["p"]:.4f}')
# ax.set_title(title, fontsize=10)
ax.legend(fontsize=9, loc='best')
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()

fig_path = os.path.join(FIGURES_DIR, 'better_choice_rate_rmANOVA.png')
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Saved: {fig_path}')